In [ ]:
from pathlib import Path

import numpy as np
from pyarrow import parquet as pq
from sbi.analysis import pairplot

from mach3sbitools.inference import InferenceHandler
from mach3sbitools.utils import MaCh3Logger

MaCh3Logger("mach3sbitools")

# Training The Algorithm
This is the fun part! We now get to train the SBI algorithm. In part 3 you made all the inputs required for this:
1. The input simulations: `data/data.feather`
2. The input prior : `priors/my_prior.pkl`

This means you're now ready to train the SBI algorithm! To do this simply run
```sh
mach3sbi train -s models/my_sbi.ckpt -d data/ -r prior/my_prior.pkl  --show_progress --max_epochs 100 --print_interval 10000
```

This will run a 100 epoch training cycle for the SBI algorithm! And save the model to `models/my_sbi.ckpt`.

It's recommended to run this on a machine with GPUs access as it will take ~1-2 hours to finish running.

Once it's trained we can run some inference!

In [ ]:
# To run inference we need to load in the model. This can be done in the CLI with mach3sbi inference BUT it's good to do here!

# Let's first load the model. This is the same as the CLI `mach3sbi inference` step
prior = Path("prior/my_prior.pkl")
model = Path("models/my_sbi.ckpt")

physics_config = Path("physics_configs/PhysicsConfig.yaml")
# Now we grab the inference
inference_handler = InferenceHandler(prior)
inference_handler.load_posterior(model, posterior_config=None)

parameter_names = inference_handler.prior.prior_data.parameter_names

In [ ]:
# We can now load in our data point
observed_data_file = Path("observations/observed_data.paraquet")
observed_data = np.array(pq.read_table(observed_data_file)["data"])

In [ ]:
# Now we can run the inference
n_samples = 1_000_000
samples = inference_handler.sample_posterior(n_samples, observed_data).cpu().numpy()

In [ ]:
# And plot the posterior as a triangle plot!
pairplot(samples, labels=[[p] for p in parameter_names])

Being able to run inference is all well and good, but we need to know if what we've done is correct! The next part will cover diagnostics